# Build our CellXGene training and validation sets

Based on what we learned in the EDA notebook, we build up training and validation sets with which to train a model

In [8]:
%run notebook_setup.ipynb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
autoreload enabled
repo_dir set to /root/GenePT-tools
File already exists at /root/GenePT-tools/data/GenePT_emebdding_v2.zip
Extracting files...
Extracting GenePT_emebdding_v2/
Skipping GenePT_emebdding_v2/NCBI_UniProt_summary_of_genes.json - already exists with same size
Skipping GenePT_emebdding_v2/GenePT_gene_embedding_ada_text.pickle - already exists with same size
Skipping GenePT_emebdding_v2/GenePT_gene_protein_embedding_model_3_text.pickle. - already exists with same size
Skipping GenePT_emebdding_v2/NCBI_summary_of_genes.json - already exists with same size
Extraction complete!
Skipping embedding_original_ada_text.parquet - already exists
Skipping embedding_original_large_3.parquet - already exists
Skipping embedding_associations_age_cell_type_drugs_pathways_openai_large.parquet - already exists
Skipping embedding_associations_age_drugs_pathways_openai_large.parquet - already exists
Skipping

## Load CellXGene V2 metadata

I extracted some metadata contained in the CellXGene V2 AnnData files using https://github.com/honicky/anndata-metadata
and saved it in `cellxgene_v2_metadata_v2.parquet`


In [9]:
import pandas as pd

metadata_pdf = pd.read_parquet(data_dir / "cellxgene_v2_metadata_v2.parquet")

In [10]:
import json

def aggregate_obs_counts(pdf, obs_count_name):
  all_obs_counts = {}
  for json_obj in pdf.obs_counts:
    obs_counts = json.loads(json_obj[obs_count_name])
    for key, value in obs_counts.items():
      if key in all_obs_counts:
        all_obs_counts[key] += value
      else:
        all_obs_counts[key] = value

  obs_counts_pdf = pd.DataFrame(all_obs_counts.items(), columns=[obs_count_name, 'count']).set_index(obs_count_name).sort_values('count', ascending=False)
  return obs_counts_pdf

cell_type_counts_pdf = aggregate_obs_counts(metadata_pdf, 'cell_type')
cell_type_counts_pdf.head(50)

,count
cell_type,
neuron,7271693
L2/3-6 intratelencephalic projecting glutamatergic neuron,4310965
oligodendrocyte,3556599
fibroblast,2267012
unknown,2159597
"CD4-positive, alpha-beta T cell",2045255
"CD8-positive, alpha-beta T cell",1952771
macrophage,1950346
classical monocyte,1591535


## Load AnnData unstructured metadata fields

We created `descriptions.parquet` in the EDA notebook.

In [11]:
descriptions_pdf = pd.read_parquet(data_dir / "cellxgene" / "descriptions.parquet")

### Double check that we can join `metadata_pdf` and `descriptions_pdf`

In [12]:
print(metadata_pdf.index)
print(descriptions_pdf.index)

cominbed_metadata_pdf = pd.concat([metadata_pdf, descriptions_pdf], axis=1)

RangeIndex(start=0, stop=961, step=1)
RangeIndex(start=0, stop=961, step=1)


In [13]:
cominbed_metadata_pdf[cominbed_metadata_pdf.published_at > '2023-05-08'].shape

(393, 48)

# selection criteria

* 10K cells for each cell type with enough
  * select uniforming across files 
* hold out 10% for testing for other types
* filter out tabula sapiens 
* test set should be past 2023-05-08 when scGPT was trained
  * 393 files
  * 152 genes in our list are not represented in these files 


In [14]:
import numpy as np

cell_counts_pdf = pd.DataFrame({
  row['cell_type']: metadata_pdf.obs_counts.apply(lambda x: json.loads(x['cell_type']).get(row['cell_type'], 0))
  for i, row in np.minimum(cell_type_counts_pdf[(cell_type_counts_pdf['count'] > 500)], 10_000).reset_index().iterrows()
})




In [15]:
sum(cell_counts_pdf[cominbed_metadata_pdf.published_at > '2023-05-08'].sum() == 0)


152

In [16]:
cell_counts_pdf[cominbed_metadata_pdf.organism.apply(lambda x: len(x) == 1 and x[0]['label'] == 'Homo sapiens')].shape


(820, 738)

## We're ready to filter the metadata and cell counts

In [17]:
human_only_x10_metadata_pdf = cominbed_metadata_pdf[
    (cominbed_metadata_pdf.organism.apply(lambda x: len(x) == 1 and x[0]['label'] == 'Homo sapiens'))
    & (cominbed_metadata_pdf.assay.apply(lambda x: all(a['label'].startswith('10x') for a in x)))
  ].reset_index(drop=True)
human_only_x10_cell_counts_pdf = cell_counts_pdf[
    (cominbed_metadata_pdf.organism.apply(lambda x: len(x) == 1 and x[0]['label'] == 'Homo sapiens'))
    & (cominbed_metadata_pdf.assay.apply(lambda x: all(a['label'].startswith('10x') for a in x)))
  ].reset_index(drop=True)

test_file_indices = human_only_x10_metadata_pdf[human_only_x10_metadata_pdf.published_at > '2023-05-08'].cell_type.apply(lambda x: len(x)).sort_values(ascending=False)[:100].index
# cominbed_metadata_pdf[cominbed_metadata_pdf.published_at > '2023-05-08'].cell_type[0]

In [18]:
human_only_x10_cell_counts_pdf.iloc[test_file_indices].shape


(100, 738)

In [19]:
human_only_x10_cell_counts_pdf.iloc[~human_only_x10_cell_counts_pdf.index.isin(test_file_indices)].head()
# test_file_indices

,neuron,L2/3-6 intratelencephalic projecting glutamatergic neuron,oligodendrocyte,fibroblast,unknown,"CD4-positive, alpha-beta T cell","CD8-positive, alpha-beta T cell",macrophage,classical monocyte,T cell,...,endothelial cell of sinusoid,epithelial cell of uterus,precursor cell,B-1a B cell,"CD34-positive, CD56-positive, CD117-positive common innate lymphoid precursor, human",ON retinal ganglion cell,kidney collecting duct cell,fibroblast of lymphatic vessel,brush cell of epithelium proper of large intestine,adipocyte of epicardial fat of left ventricle
0,9022,0,635,25,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,5399,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,158,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,19391,6532,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Restructure so that we can pull out the right number of cells per file

In [20]:
training_counts = []

for cell_type in human_only_x10_cell_counts_pdf.columns:
    if cell_type == 'unknown':
        continue
    # Exclude test indices for training
    train_df = human_only_x10_cell_counts_pdf.iloc[~human_only_x10_cell_counts_pdf.index.isin(test_file_indices)]
    # Only keep rows where this cell type has > 0 cells
    counts_pdf = train_df[train_df[cell_type] > 0][[cell_type]].reset_index(drop=False)
    counts_pdf.columns = ['index', 'cell_count']
    counts_pdf['cell_type'] = cell_type

    total_cell_count = counts_pdf['cell_count'].sum()
    num_rows = len(counts_pdf)

    if total_cell_count <= 10000:
        # If under or equal to 10,000, take all
        selected = counts_pdf.copy()
    else:
        # Evenly divide 10,000 among rows
        base_share = 10000 // num_rows
        remainder = 10000 % num_rows

        # Assign base_share to each row, and distribute the remainder
        counts_pdf['take_cells'] = base_share
        if remainder > 0:
            counts_pdf.loc[counts_pdf.index[:remainder], 'take_cells'] += 1

        # For each row, take up to the minimum of take_cells and available cells
        counts_pdf['cell_count'] = counts_pdf[['take_cells', 'cell_count']].min(axis=1)

        # If some rows can't provide their full share, redistribute the leftover
        leftover = 10000 - counts_pdf['cell_count'].sum()
        while leftover > 0:
            # Find rows that still have available cells
            mask = counts_pdf['cell_count'] < train_df.loc[counts_pdf['index'], cell_type].values
            if not mask.any():
                break  # No more cells to take
            for idx in counts_pdf[mask].index:
                available = train_df.loc[counts_pdf.at[idx, 'index'], cell_type]
                if counts_pdf.at[idx, 'cell_count'] < available:
                    counts_pdf.at[idx, 'cell_count'] += 1
                    leftover -= 1
                    if leftover == 0:
                        break

        selected = counts_pdf[['index', 'cell_type', 'cell_count']]

    training_counts.append(selected)

In [21]:
training_counts_pdf = pd.concat(training_counts)
training_counts_pdf

,index,cell_type,cell_count
0,0,neuron,68
1,13,neuron,68
2,16,neuron,68
3,18,neuron,68
4,32,neuron,68
...,...,...,...
2,368,"CD34-positive, CD56-positive, CD117-positive c...",395
0,579,ON retinal ganglion cell,549
0,471,kidney collecting duct cell,420
1,717,kidney collecting duct cell,125


In [22]:
human_only_x10_metadata_pdf[human_only_x10_metadata_pdf.filename.str.contains("ff4cfa86-9c0c-4b7c-abd6-90547657d04f")]

,file_size,main_groups,cell_count,gene_count,obs_contents,var_contents,x_storage,embeddings,pairwise_relationships,expression_layers,...,self_reported_ethnicity,sex,spatial,suspension_type,tissue,title,tombstone,visibility,x_approximate_distribution,batch_condition
741,246620277,"[X, layers, obs, obsm, obsp, uns, var, varm, v...",9799,24855,"[Phase, _index, assay, assay_ontology_term_id,...","[feature_biotype, feature_is_filtered, feature...","{'chunk_size': [20497], 'components': ['data',...","[X_pca, X_umap]",[],[lognorm],...,"[{'label': 'European', 'ontology_term_id': 'HA...","[{'label': 'female', 'ontology_term_id': 'PATO...",None,[nucleus],"[{'label': 'chest wall', 'ontology_term_id': '...",HTAPP-213-SMP-6752 scRNA-seq,False,PUBLIC,None,None


## Optimize the amount of data we load from S3

* `select_row_groups_ilp` picks the minimum set of row groups to get the required cell type counts from a parquet file.  It uses an Integer Linear Programming solver to find an optimal solution quickly
* use `s3fs` to access only portions of `s3` objects that are needed to read the rows and columns needed
* use `arrow` to efficiently access specific portions of the parquet file without loading the entire file  

In [24]:
import pulp

def select_row_groups_ilp(cell_type_counts_per_row_group, needed):
  """
  cell_type_counts_per_row_group: list of dicts, one per row group, mapping cell_type to count
  needed: dict mapping cell_type to required count
  Returns: set of row group indices to read
  """
  n_row_groups = len(cell_type_counts_per_row_group)
  cell_types = list(needed.keys())

  # Define the problem
  prob = pulp.LpProblem("RowGroupSelection", pulp.LpMinimize)

  # Binary variables: x_r = 1 if row group r is selected
  x = [pulp.LpVariable(f"x_{r}", cat="Binary") for r in range(n_row_groups)]

  # Objective: minimize number of row groups
  prob += pulp.lpSum(x)

  # Constraints: for each cell type, enough cells must be selected
  for c in cell_types:
    prob += (
      pulp.lpSum(
        cell_type_counts_per_row_group[r].get(c, 0) * x[r]
        for r in range(n_row_groups)
      ) >= needed[c],
      f"cover_{c}"
    )

  # Solve
  prob.solve()

  # Extract selected row groups
  selected = {r for r in range(n_row_groups) if pulp.value(x[r]) > 0.5}
  return selected

In [25]:
# Test case 1: Simple, non-overlapping
cell_type_counts_per_row_group_1 = [
  {'A': 10},   # row group 0
  {'B': 10},   # row group 1
  {'C': 10},   # row group 2
]
needed_1 = {'A': 5, 'B': 5, 'C': 5}
print("Test 1 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_1, needed_1))
# Expected: {0, 1, 2}

# Test case 2: Overlapping, can use fewer row groups
cell_type_counts_per_row_group_2 = [
  {'A': 5, 'B': 5},   # row group 0
  {'B': 5, 'C': 5},   # row group 1
  {'A': 5, 'C': 5},   # row group 2
]
needed_2 = {'A': 5, 'B': 5, 'C': 5}
print("Test 2 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_2, needed_2))
# Expected: Any two row groups, e.g., {0,1}, {0,2}, or {1,2}

# Test case 3: Need more than one row group for a cell type
cell_type_counts_per_row_group_3 = [
  {'A': 3},   # row group 0
  {'A': 2, 'B': 5},   # row group 1
  {'B': 5, 'C': 5},   # row group 2
]
needed_3 = {'A': 5, 'B': 5}
print("Test 3 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_3, needed_3))
# Expected: {0,1} (for A), and either 1 or 2 for B, so {0,1}, {0,2}, or {0,1,2}

# Test case 4: Impossible to satisfy
cell_type_counts_per_row_group_4 = [
  {'A': 2},   # row group 0
  {'B': 2},   # row group 1
]
needed_4 = {'A': 5, 'B': 5}
print("Test 4 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_4, needed_4))
# Expected: set(), or infeasible (no solution)

# Test case 5: Redundant row group
cell_type_counts_per_row_group_5 = [
  {'A': 5, 'B': 5},   # row group 0
  {'A': 5},           # row group 1
  {'B': 5},           # row group 2
]
needed_5 = {'A': 5, 'B': 5}
print("Test 5 selected row groups:", select_row_groups_ilp(cell_type_counts_per_row_group_5, needed_5))
# Expected: {0}

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /root/GenePT-tools/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/a86a7294a9f7412aab543c5a1b47ace8-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/a86a7294a9f7412aab543c5a1b47ace8-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 8 COLUMNS
At line 21 RHS
At line 25 BOUNDS
At line 29 ENDATA
Problem MODEL has 3 rows, 3 columns and 3 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1.5 - 0.00 seconds
Cgl0004I processed model has 0 rows, 0 columns (0 integer (0 of which binary)) and 0 elements
Cbc3007W No integer variables - nothing to do
Cuts at root node changed objective from 3 to -1.79769e+308
Probing was tried 0 times and created 0 cuts of which 0 were active after adding rounds of cuts (0.000 seconds)
Gomory was tried 0 times and creat

In [26]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import tqdm
import s3fs
from collections import defaultdict
import pyarrow.compute as pc
import numpy as np

fs = s3fs.S3FileSystem(anon=False, profile='xcellerate')


def optimized_row_group_sample_arrow(counts_df: pd.DataFrame, parquet_path: str) -> pd.DataFrame:
  """
  Efficiently sample rows for each cell type by first scanning row groups for relevant cell types and their counts,
  then only reading the minimal set of row groups needed to fulfill the sample requirements.
  Uses Arrow for filtering to minimize memory usage.
  """
  needed = {
    row['cell_type']: row['cell_count']
    for _, row in counts_df.iterrows()
  }
  collected = {cell_type: [] for cell_type in needed}
  done = set()

  print(f"Opening {parquet_path}")
  with (
    fs.open(parquet_path, 'rb')
    if parquet_path[:5] == "s3://"
    else open(parquet_path, 'rb')
  ) as f:
    parquet_file = pq.ParquetFile(f)
    n_row_groups = parquet_file.num_row_groups
    print(f"Found {n_row_groups} row groups")

    if n_row_groups > 1:
      # First pass: build a list of dicts with cell type counts for each row group
      print("Indexing row groups for cell types and counts...")
      cell_type_counts_per_row_group = []
      for rg_idx in tqdm.tqdm(range(n_row_groups)):
        table = parquet_file.read_row_group(
          rg_idx,
          columns=['cell_type', 'assay', 'organism']
        )
        # Arrow filtering
        assay_str = pc.cast(table['assay'], pa.string())
        organism_str = pc.cast(table['organism'], pa.string())
        assay_mask = pc.starts_with(assay_str, '10x')
        organism_mask = pc.equal(organism_str, 'Homo sapiens')
        combined_mask = pc.and_(assay_mask, organism_mask)
        filtered_table = table.filter(combined_mask)
        counts = filtered_table['cell_type'].to_pandas().value_counts().to_dict()
        cell_type_counts_per_row_group.append(counts)
        del table, filtered_table

      # Use ILP to select the optimal set of row groups
      print("Selecting optimal row groups using ILP...")
      row_groups_to_read = select_row_groups_ilp(cell_type_counts_per_row_group, needed)
      del cell_type_counts_per_row_group
      row_groups_to_read = sorted(row_groups_to_read)
    else:
      row_groups_to_read = [ 0 ]

    # Second pass: read only the needed row groups, and sample for each cell type
    print(f"Reading {len(row_groups_to_read)} row groups ...")
    for rg_idx in tqdm.tqdm(row_groups_to_read):
      print("reading row group", rg_idx)
      num_rows = parquet_file.metadata.row_group(rg_idx).num_rows
      print("num rows", num_rows)
      table = parquet_file.read_row_group(
        rg_idx,
        columns=['cell_type', 'assay', 'organism'] + [
          col for col in parquet_file.schema.names
          if col not in ['cell_type', 'assay', 'organism']
        ]
      )

      print("filtering")

      # Arrow filtering
      assay_str = pc.cast(table['assay'], pa.string())
      organism_str = pc.cast(table['organism'], pa.string())
      assay_mask = pc.starts_with(assay_str, '10x')
      organism_mask = pc.equal(organism_str, 'Homo sapiens')
      combined_mask = pc.and_(assay_mask, organism_mask)
      filtered_table = table.filter(combined_mask)
      del table

      # For each cell type still needing samples and present in this row group
      for cell_type in needed:
        if cell_type in done:
          continue
        print("filtering for cell type", cell_type)
        cell_type_mask = pc.equal(filtered_table['cell_type'], cell_type)
        subset_table = filtered_table.filter(cell_type_mask)
        n_needed = needed[cell_type] - sum(len(x) for x in collected[cell_type])
        if n_needed <= 0:
          done.add(cell_type)
          continue
        if subset_table.num_rows > 0:
          n_sample = min(n_needed, subset_table.num_rows)
          # Sample indices in Arrow, then convert only the sample to pandas
          indices = np.random.choice(subset_table.num_rows, n_sample, replace=False)
          sampled_table = subset_table.take(indices)
          sampled_df = sampled_table.to_pandas()
          collected[cell_type].append(sampled_df)
          if n_sample == n_needed:
            done.add(cell_type)
        del subset_table
      # Stop early if all cell types are done
      if len(done) == len(needed):
        print(f"All cell types are done")
        break

  # Concatenate all collected samples
  all_samples = [pd.concat(collected[cell_type], ignore_index=True) for cell_type in collected if collected[cell_type]]
  return pd.concat(all_samples, ignore_index=True)

### Review and example optimization

In [27]:

human_only_x10_metadata_pdf.iloc[741].filename

's3://cdiam-h5ad-database/cellxgene_v2/ff4cfa86-9c0c-4b7c-abd6-90547657d04f'

In [28]:
from pathlib import Path

index = 741
filename = Path(human_only_x10_metadata_pdf.iloc[index].filename).stem
embedding_path = f"s3://pythiomicsdata/cellxgene_v2/genept_embeddings_v1/{filename}.parquet"

example_training_embeddings_pdf = optimized_row_group_sample_arrow(training_counts_pdf[training_counts_pdf["index"] == index], embedding_path)
example_training_embeddings_pdf.cell_type.value_counts()

Opening s3://pythiomicsdata/cellxgene_v2/genept_embeddings_v1/ff4cfa86-9c0c-4b7c-abd6-90547657d04f.parquet
Found 1 row groups
Reading 1 row groups ...


  0%|          | 0/1 [00:00<?, ?it/s]

reading row group 0
num rows 9799
filtering
filtering for cell type fibroblast
filtering for cell type macrophage
filtering for cell type T cell
filtering for cell type malignant cell
filtering for cell type blood vessel endothelial cell
filtering for cell type blood vessel smooth muscle cell
filtering for cell type cell of skeletal muscle


  0%|          | 0/1 [00:04<?, ?it/s]

All cell types are done


cell_type
cell of skeletal muscle            1578
malignant cell                      750
blood vessel endothelial cell       536
T cell                              173
macrophage                          134
fibroblast                           68
blood vessel smooth muscle cell       6
Name: count, dtype: int64

In [29]:
training_counts_pdf[training_counts_pdf["index"] == 741]

,index,cell_type,cell_count
180,741,fibroblast,68
75,741,macrophage,134
63,741,T cell,173
14,741,malignant cell,750
23,741,blood vessel endothelial cell,536
6,741,blood vessel smooth muscle cell,6
2,741,cell of skeletal muscle,1578


In [30]:
from pathlib import Path

for index, group in training_counts_pdf.groupby('index'):
  filename = Path(human_only_x10_metadata_pdf.iloc[index].filename)
  print(filename)
  print(group)
  break

s3:/cdiam-h5ad-database/cellxgene_v2/00476f9f-ebc1-4b72-b541-32f912ce36ea
   index                               cell_type  cell_count
0      0                                  neuron          68
0      0                         oligodendrocyte          75
0      0                              fibroblast          25
0      0                        endothelial cell          16
0      0                               astrocyte          81
0      0          oligodendrocyte precursor cell          79
0      0                                pericyte           9
0      0       central nervous system macrophage          94
0      0  vascular associated smooth muscle cell          11
0      0                               leukocyte           3
0      0                          ependymal cell          27


## Ok, GO!

Extact the embeddings from the training set and write them to a local directory

In [32]:
import s3fs
import tqdm
from dataclasses import dataclass

@dataclass
class EmbeddingFetchError:
  index: int
  file_path: str
  error: Exception
  group: pd.DataFrame

import json

error_log_path = data_dir / "cellxgene_embeddings" / "training_v1" / "embedding_errors.log"
with open(error_log_path, "a") as error_log:
  fs = s3fs.S3FileSystem(anon=False, profile='xcellerate')
  for index, group in tqdm.tqdm(training_counts_pdf.groupby('index')):
    filename = Path(human_only_x10_metadata_pdf.iloc[index].filename).stem
    file_size = human_only_x10_metadata_pdf.iloc[index].file_size

    print(f"Processing {filename} with size {file_size}")
    
    embedding_path = f"s3://pythiomicsdata/cellxgene_v2/genept_embeddings_v1/{filename}.parquet"
    
    output_path = data_dir / "cellxgene_embeddings" / "training_v1" / f"{filename}.parquet"
    if output_path.exists():
      continue

    try:
      training_embeddings_pdf = optimized_row_group_sample_arrow(
        training_counts_pdf[training_counts_pdf["index"] == index],
        embedding_path
      )
      training_embeddings_pdf.to_parquet(output_path)
      del training_embeddings_pdf
    except Exception as e:
      efe = EmbeddingFetchError(index, embedding_path, e, group)
      # Write error info as JSON (convert group to string for simplicity)
      error_log.write(json.dumps({
        "index": efe.index,
        "file_path": efe.file_path,
        "error": str(efe.error),
        "group": efe.group.reset_index(drop=True).to_json()  # or str(efe.group)
      }) + "\n")
      error_log.flush()  # Ensure it's written immediately
      print(efe)
      



  0%|          | 0/642 [00:00<?, ?it/s]

 55%|█████▌    | 354/642 [00:00<00:00, 3537.75it/s]

Processing 00476f9f-ebc1-4b72-b541-32f912ce36ea with size 750437761
Processing 0087cde2-967d-4f7c-8e6e-40e4c9ad1891 with size 389395627
Processing 00e5dedd-b9b7-43be-8c28-b0e5c6414a62 with size 705366728
Processing 00ff600e-6e2e-4d76-846f-0eec4f0ae417 with size 7187747
Processing 01209dce-3575-4bed-b1df-129f57fbc031 with size 801497925
Processing 0129dbd9-a7d3-4f6b-96b9-1da155a93748 with size 9828162445
Processing 019c7af2-c827-4454-9970-44d5e39ce068 with size 491650252
Processing 01ad3cd7-3929-4654-84c0-6db05bd5fd59 with size 12326224885
Processing 01c93cf6-b695-4e30-a26e-121ae8b16a9e with size 119252466
Processing 030faa69-ff79-4d85-8630-7c874a114c19 with size 2983375726
Processing 03181d87-4769-41e7-8c39-d9a81835f0d2 with size 4697014766
Processing 0325478a-9b52-45b5-b40a-2e2ab0d72eb1 with size 38214116935
Processing 0374f03c-62e2-4859-8a14-acb00b0627d5 with size 1043486165
Processing 03c544fb-a103-4d18-9230-eae9cfee3af2 with size 926053492
Processing 03d38670-1444-4001-bc53-9936e61

100%|██████████| 28/28 [00:27<00:00,  1.00it/s]t/s]


Selecting optimal row groups using ILP...
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /root/GenePT-tools/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/93c3d18d78fa4806bea0514bbf7ae713-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/93c3d18d78fa4806bea0514bbf7ae713-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 23 COLUMNS
At line 612 RHS
At line 631 BOUNDS
At line 660 ENDATA
Problem MODEL has 18 rows, 28 columns and 504 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 28 - 0.00 seconds
Cgl0004I processed model has 0 rows, 0 columns (0 integer (0 of which binary)) and 0 elements
Cbc3007W No integer variables - nothing to do
Cuts at root node changed objective from 28 to -1.79769e+308
Probing was tried 0 times and created 0 cuts of which 0 were active after adding rounds of cuts 

...

Processing ff7d15fa-f4b6-4a0e-992e-fd0c9d088ded with size 2013071830
Opening s3://pythiomicsdata/cellxgene_v2/genept_embeddings_v1/ff7d15fa-f4b6-4a0e-992e-fd0c9d088ded.parquet
Found 1 row groups
Reading 1 row groups ...


reading row group 0
num rows 28051
filtering
filtering for cell type neuron
filtering for cell type oligodendrocyte
filtering for cell type fibroblast
filtering for cell type endothelial cell
filtering for cell type astrocyte
filtering for cell type oligodendrocyte precursor cell
filtering for cell type pericyte
filtering for cell type central nervous system macrophage
filtering for cell type vascular associated smooth muscle cell


  0%|          | 0/1 [00:13<?, ?it/s]

filtering for cell type leukocyte
All cell types are done



100%|██████████| 642/642 [1:36:45<00:00,  9.04s/it]


In [376]:
embedding_path = "s3://pythiomicsdata/cellxgene_v2/genept_embeddings_v1/0ae6f031-2f9c-4247-8b26-db320d6efd32.parquet"

with fs.open(embedding_path, 'rb') as f:
  embeddings_pdf = pd.read_parquet(f)


In [ ]:
embeddings_pdf.assay.value_counts()